In [1]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from IPython.display import clear_output
from defs import normalize_matrix, ST_GNN, rPPGWindow, bandpass, neg_pearson_loss
import cv2
from scipy.interpolate import interp1d
import torch 
import torch.nn as nn
import torch.optim as optim


In [2]:
import mediapipe as mp

In [3]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")


ds = load_dataset("kyegorov/mcd_rppg", streaming=True, split="train", token=hf_token)

ds

Resolving data files:   0%|          | 0/12002 [00:00<?, ?it/s]

IterableDataset({
    features: Unknown,
    num_shards: 7201
})

In [4]:
from huggingface_hub import hf_hub_download
import pandas as pd

# 1. Download the master mapping file
print("Downloading db.csv...")
csv_path = hf_hub_download(repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename="db.csv")
df = pd.read_csv(csv_path)

print(df.head())


target_video = df['video'][100]

print(f"Downloading {target_video}...")
video_path = hf_hub_download(repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename=target_video)

video_path

   patient_id  weight  height        bmi   age sex  upper_ap  lower_ap  \
0        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
1        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
2        1020    55.0   170.0  19.031142  23.0   F     113.0      78.0   
3        1020    55.0   170.0  19.031142  23.0   F     105.0      78.0   
4        1020    55.0   170.0  19.031142  23.0   F     105.0      78.0   

   saturation  temperature  ...  pulse  stress    step        camera   view  \
0        98.0         36.6  ...  100.0     4.0   after  FullHDwebcam  front   
1        98.0         36.6  ...  100.0     4.0   after      USBVideo   left   
2        98.0         36.6  ...  100.0     4.0   after   IriunWebcam  right   
3        99.0         36.6  ...   83.0     4.0  before  FullHDwebcam  front   
4        99.0         36.6  ...   83.0     4.0  before      USBVideo   left   

                    ecg                 ppg  \
0   ecg/1020_after.json   ppg/102

'/Users/ihsanbolum/.cache/huggingface/hub/datasets--kyegorov--mcd_rppg/snapshots/49d81770d85e024a529800252530c553e52cdd44/video/1234_USBVideo_before.avi'

In [5]:
import cv2
import mediapipe as mp
import numpy as np
from tqdm.notebook import tqdm
FOREHEAD_NODES = [10, 151, 67, 109, 108, 107, 297, 338, 336, 337]
# Cheeks: fleshy mid-cheek, off the nose and away from mouth corners
CHEEK_NODES    = [50, 205, 187, 123, 116, 280, 425, 411, 352, 345]
SELECTED_NODES = FOREHEAD_NODES + CHEEK_NODES
N_NODES = len(SELECTED_NODES)

face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)



Z_OCCLUSION_MARGIN = 0.07


def extract_frame_features(frame, out):
    """Fills out (N_NODES, 4) in-place with [x, y, avg_green, valid] per node.

    valid = 1 if the landmark is on the camera-facing side AND the patch fits in frame,
    valid = 0 (and other channels zeroed) otherwise.
    """
    h, w, _ = frame.shape
    results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    out.fill(0)
    if not results.multi_face_landmarks:
        return  # no face -> all zeros, valid=0 everywhere

    landmarks = results.multi_face_landmarks[0].landmark

    # Occlusion via z: far-side landmarks have larger z than the front-most node.
    zs = np.array([landmarks[idx].z for idx in SELECTED_NODES], dtype=np.float32)
    z_front = zs.min()
    occluded = zs > z_front + Z_OCCLUSION_MARGIN

    for i, idx in enumerate(SELECTED_NODES):
        if occluded[i]:
            continue  # leave as zeros, valid stays 0
        pt = landmarks[idx]
        px, py = int(pt.x * w), int(pt.y * h)
        if not (2 <= px < w - 2 and 2 <= py < h - 2):
            continue  # patch off-frame -> treat as invalid
        out[i, 0] = pt.x
        out[i, 1] = pt.y
        out[i, 2] = frame[py-2:py+3, px-2:px+3, 1].mean()
        out[i, 3] = 1.0  # valid


def extract_graph_features(video_path):
    output_name = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = os.path.join(os.getcwd(), "final_matrix_video")
    save_path = os.path.join(save_dir, output_name + ".npy")
    if os.path.exists(save_path):
        return "already done"

    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Pre-allocate when total_frames is known; otherwise grow in chunks.
    if total_frames > 0:
        buf = np.empty((total_frames, N_NODES, 4), dtype=np.float32)
        idx = 0
        pbar = tqdm(total=total_frames, desc=output_name, unit="frame")
        while True:
            success, frame = cap.read()
            if not success:
                break
            if idx >= total_frames:
                buf = np.concatenate([buf, np.empty((256, N_NODES, 4), dtype=np.float32)], axis=0)
            extract_frame_features(frame, buf[idx])
            idx += 1
            pbar.update(1)
        pbar.close()
        final_matrix = buf[:idx]
    else:
        chunks = []
        chunk = np.empty((512, N_NODES, 4), dtype=np.float32)
        idx = 0
        pbar = tqdm(desc=output_name, unit="frame")
        while True:
            success, frame = cap.read()
            if not success:
                break
            if idx == chunk.shape[0]:
                chunks.append(chunk)
                chunk = np.empty((512, N_NODES, 4), dtype=np.float32)
                idx = 0
            extract_frame_features(frame, chunk[idx])
            idx += 1
            pbar.update(1)
        pbar.close()
        chunks.append(chunk[:idx])
        final_matrix = np.concatenate(chunks, axis=0)

    cap.release()
    np.save(save_path, final_matrix)
    print(f"Saved: {save_path}  Shape: {final_matrix.shape}")
    return final_matrix




I0000 00:00:1778147502.214157 2778594 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778147502.227862 2779012 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778147502.232184 2779011 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [21]:
# Visual sanity check: show ONLY the landmarks that pass the same valid/occlusion
# checks used by extract_frame_features. Tweak Z_OCCLUSION_MARGIN above and re-run.
import cv2
from IPython.display import display, clear_output
from PIL import Image







PATCH_HALF = 2  # matches frame[py-2:py+3, px-2:px+3] in extract_frame_features

def preview_roi(video_path, max_frames=300, stride=2, show_occluded=True):
    """show_occluded=True draws masked-out points as a faint red X for debugging.
    Set False to see only what the GNN will actually receive."""
    shown = 0
    try:
        while cap.isOpened() and shown < max_frames:
            success, frame = cap.read()
            if not success:
                break
            if shown % stride != 0:
                shown += 1
                continue

            h, w, _ = frame.shape
            results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            vis = frame.copy()

            if results.multi_face_landmarks:
                lms = results.multi_face_landmarks[0].landmark

                # Same occlusion logic as extract_frame_features
                zs = np.array([lms[idx].z for idx in SELECTED_NODES], dtype=np.float32)
                z_front = zs.min()
                occluded = zs > z_front + Z_OCCLUSION_MARGIN

                n_valid = 0
                for i, idx in enumerate(SELECTED_NODES):
                    px = int(lms[idx].x * w)
                    py = int(lms[idx].y * h)
                    in_frame = 2 <= px < w - 2 and 2 <= py < h - 2
                    is_valid = (not occluded[i]) and in_frame

                    if is_valid:
                        color = (0, 255, 0) if i < len(FOREHEAD_NODES) else (0, 200, 255)
                        cv2.rectangle(vis,
                                      (px - PATCH_HALF, py - PATCH_HALF),
                                      (px + PATCH_HALF, py + PATCH_HALF),
                                      color, 1)
                        cv2.putText(vis, str(idx), (px + 4, py - 4),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.3, color, 1, cv2.LINE_AA)
                        n_valid += 1
                    elif show_occluded:
                        # Faint red X for masked nodes (occluded or off-frame)
                        cv2.line(vis, (px - 3, py - 3), (px + 3, py + 3), (0, 0, 180), 1)
                        cv2.line(vis, (px - 3, py + 3), (px + 3, py - 3), (0, 0, 180), 1)

                cv2.putText(vis,
                            f"valid {n_valid}/{N_NODES}  margin={Z_OCCLUSION_MARGIN:.2f}",
                            (10, 25),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                cv2.putText(vis, "NO FACE DETECTED", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            clear_output(wait=True)
            display(Image.fromarray(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)))
            shown += 1
    except KeyboardInterrupt:
        print("Stopped by user.")
    finally:
        cap.release()


preview_roi(video_path, max_frames=300, stride=2, show_occluded=True)


In [ ]:
import threading
import queue

def download_extract_video():
    videos = df["video"].drop_duplicates().tolist()
    save_dir = os.path.join(os.getcwd(), "final_matrix_video")
    os.makedirs(save_dir, exist_ok=True)

    # Skip already-processed up front so we don't queue them.
    todo = []
    for v in videos:
        out_name = os.path.splitext(os.path.basename(v))[0] + ".npy"
        if not os.path.exists(os.path.join(save_dir, out_name)):
            todo.append(v)
    print(f"{len(todo)} videos to process ({len(videos) - len(todo)} already done)")

    q = queue.Queue(maxsize=2)  # 1 ready + 1 being downloaded
    SENTINEL = object()

    def producer():
        for v in todo:
            try:
                path = hf_hub_download(
                    repo_id="kyegorov/mcd_rppg", repo_type="dataset", filename=v
                )
                q.put(path)
            except Exception as e:
                print(f"  download error {v}: {e}")
        q.put(SENTINEL)

    t = threading.Thread(target=producer, daemon=True)
    t.start()

    for i in range(len(todo)):
        path = q.get()
        if path is SENTINEL:
            break
        try:
            print(f"[{i+1}/{len(todo)}] extracting {os.path.basename(path)}")
            extract_graph_features(path)
        except Exception as e:
            print(f"  extract error {path}: {e}")
        finally:
            try:
                os.remove(path)  # free disk
            except OSError:
                pass

    t.join()


download_extract_video()

In [6]:
raw_dir = "final_matrix_video"
norm_dir = "final_matrix_video_norm"
os.makedirs(norm_dir, exist_ok=True)

for dateiname in os.listdir(raw_dir):
    if not dateiname.endswith(".npy"):
        continue
    out_path = os.path.join(norm_dir, dateiname)
    if os.path.exists(out_path):
        continue  

    matrix = np.load(os.path.join(raw_dir, dateiname))
    matrix = normalize_matrix(matrix)
    np.save(out_path, matrix)


In [7]:


def sync_matrix_ppg(matrix_path, ppg_path):
    try:

        ppg = np.loadtxt(ppg_path)
        if ppg.ndim > 1:
            ppg = ppg[:, 0] 
            
        matrix = np.load(matrix_path)

        target_len = matrix.shape[0]
        current_len = ppg.shape[0]

        if target_len != current_len:
            x_old = np.linspace(0, 1, current_len)
            x_new = np.linspace(0, 1, target_len)
            ppg = interp1d(x_old, ppg, kind="linear")(x_new)

        ppg = ppg.astype(np.float32)                                                                                                                                                                            
        ppg = bandpass(ppg, fs=30.0)                                                                                                                                                                            
        ppg = (ppg - ppg.mean()) / (ppg.std() + 1e-8)


        return ppg
        
    except FileNotFoundError:
        return None 



output_folder = "ppg_sync_final"
os.makedirs(output_folder, exist_ok=True)

for matrix_file in sorted(os.listdir("final_matrix_video_norm")):
    if matrix_file.endswith(".npy"):
        rppg_file = matrix_file.replace(".npy", ".txt")


        v_path = os.path.join("final_matrix_video_norm", matrix_file)
        p_path = os.path.join("ppg_sync", rppg_file)

        synced_ppg = sync_matrix_ppg(v_path, p_path)

        if synced_ppg is not None:
            save_name = rppg_file.replace(".txt", ".npy")
            save_path = os.path.join(output_folder, save_name)
            
            np.save(save_path, synced_ppg)
            print(f"✅ Gespeichert: {save_name} (Länge: {synced_ppg.shape[0]})")
        else:
            print(f"❌ Übersprungen: Keine PPG-Datei gefunden für {matrix_file}")

✅ Gespeichert: 1020_FullHDwebcam_after.npy (Länge: 5383)
✅ Gespeichert: 1020_FullHDwebcam_before.npy (Länge: 5382)
✅ Gespeichert: 1020_IriunWebcam_after.npy (Länge: 4290)
✅ Gespeichert: 1020_IriunWebcam_before.npy (Länge: 4310)
✅ Gespeichert: 1020_USBVideo_after.npy (Länge: 5377)
✅ Gespeichert: 1020_USBVideo_before.npy (Länge: 5377)
✅ Gespeichert: 1024_FullHDwebcam_after.npy (Länge: 5362)
✅ Gespeichert: 1024_FullHDwebcam_before.npy (Länge: 5365)
✅ Gespeichert: 1024_IriunWebcam_after.npy (Länge: 4309)
✅ Gespeichert: 1024_IriunWebcam_before.npy (Länge: 4285)
✅ Gespeichert: 1024_USBVideo_after.npy (Länge: 5356)
✅ Gespeichert: 1024_USBVideo_before.npy (Länge: 5376)
✅ Gespeichert: 1035_FullHDwebcam_after.npy (Länge: 5385)
✅ Gespeichert: 1035_FullHDwebcam_before.npy (Länge: 5352)
✅ Gespeichert: 1035_IriunWebcam_after.npy (Länge: 4272)
✅ Gespeichert: 1035_IriunWebcam_before.npy (Länge: 3787)
✅ Gespeichert: 1035_USBVideo_after.npy (Länge: 5365)
✅ Gespeichert: 1035_USBVideo_before.npy (Länge: 5

ai tarin / test / val split

In [13]:
import re
from sklearn.model_selection import train_test_split


def train_test_val_split(folder):
    train = []

    val = []
    def extract_id(list):
        list = pd.Series(list)
        ids = list.str.extract((r'^(\d+)'))[0]
        return ids.unique().astype(int)

    unique_ids = extract_id(folder)
    train_ids, val_ids = train_test_split(unique_ids, test_size=0.3, random_state=42)
    #val_ids = train_test_split(temp_ids, test_size=1, random_state=42)

    for i in folder:
        number_in_filName = int(i.split('_')[0])
        if number_in_filName in train_ids:
            train.append(i)
        elif number_in_filName in val_ids:
            val.append(i)

    return train, val


train, val = train_test_val_split(os.listdir("final_matrix_video_norm"))

print(len(train), len(val))

1734 741


In [14]:
from torch.utils.data import DataLoader


v_folder = "final_matrix_video_norm"
p_folder = "ppg_sync_final"
train_files, val_files, test_files = train, val, test


train_dataset = rPPGWindow(train_files, v_folder, p_folder, window_size=256, stride=128)
val_dataset   = rPPGWindow(val_files, v_folder, p_folder, window_size=256, stride=128) 

# 2. Die DataLoaders (Die Liefermaschinen)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)  

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


features = 3 
hidden_dim = 64 
num_nodes = 20

model = ST_GNN(in_features=features, hidden_dim=hidden_dim, num_nodes=num_nodes)
model = model.to(device)

criterion = nn.MSELoss(reduction='none') # check why reduction='none'

optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_inx, (x,mask,y) in enumerate(train_loader):

        x,mask,y = x.to(device),mask.to(device), y.to(device)

        optimizer.zero_grad()

        predictions = model(x, mask)

        frame_valid =(mask.sum(dim=2) >= 1).float()
        err = neg_pearson_loss(predictions, y, mask=frame_valid)

        loss = neg_pearson_loss(predictions, y, mask=frame_valid)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()                                                                                                                                                                        


    avg_train_loss = train_loss / len(train_loader)
    model.eval()

    val_loss = 0.0

    with torch.no_grad():
        for x,mask, y in val_loader:
            x,mask, y = x.to(device), mask.to(device), y.to(device)

            predictions = model(x, mask)
            frame_valid = (mask.sum(dim=2) >= 1).float()
            err = neg_pearson_loss(predictions, y, mask=frame_valid) 


            val_loss += neg_pearson_loss(predictions, y, mask=frame_valid).item() 

        avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
torch.save(model.state_dict(), "model.pt")                                                                                                                                                 

Epoch [01/20] | Train Loss: 0.7590 | Val Loss: 0.7602
Epoch [02/20] | Train Loss: 0.7234 | Val Loss: 0.7057
Epoch [03/20] | Train Loss: 0.6757 | Val Loss: 0.6892
Epoch [04/20] | Train Loss: 0.6654 | Val Loss: 0.6821
Epoch [05/20] | Train Loss: 0.6600 | Val Loss: 0.6809
Epoch [06/20] | Train Loss: 0.6556 | Val Loss: 0.6774
Epoch [07/20] | Train Loss: 0.6521 | Val Loss: 0.6782
Epoch [08/20] | Train Loss: 0.6495 | Val Loss: 0.6696
Epoch [09/20] | Train Loss: 0.6472 | Val Loss: 0.6675
Epoch [10/20] | Train Loss: 0.6456 | Val Loss: 0.6708
Epoch [11/20] | Train Loss: 0.6437 | Val Loss: 0.6710
Epoch [12/20] | Train Loss: 0.6424 | Val Loss: 0.6653
Epoch [13/20] | Train Loss: 0.6412 | Val Loss: 0.6642
Epoch [14/20] | Train Loss: 0.6404 | Val Loss: 0.6628
Epoch [15/20] | Train Loss: 0.6391 | Val Loss: 0.6604
Epoch [16/20] | Train Loss: 0.6384 | Val Loss: 0.6604
Epoch [17/20] | Train Loss: 0.6373 | Val Loss: 0.6625
Epoch [18/20] | Train Loss: 0.6368 | Val Loss: 0.6603
Epoch [19/20] | Train Loss: 

In [18]:
import cv2, numpy as np, torch, time
from defs import normalize_matrix, bandpass, ST_GNN

# Self-contained model load. Values must match what trained model.pt.
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
features   = 3
hidden_dim = 64
num_nodes  = 20
model = ST_GNN(in_features=features, hidden_dim=hidden_dim, num_nodes=num_nodes).to(device)
model.load_state_dict(torch.load("model.pt", map_location=device))
model.eval()

WIN          = 256
INFER_EVERY  = 10
MAX_FRAMES   = 2000

buf = np.zeros((WIN, N_NODES, 4), dtype=np.float32)
sig_disp = np.zeros(WIN, dtype=np.float32)
idx = 0
bpm = 0.0
measured_fps = 30.0
fps_times = []

cap = cv2.VideoCapture(0)
model.eval()

try:
    while idx < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break

        feat = np.zeros((N_NODES, 4), dtype=np.float32)
        extract_frame_features(frame, feat)
        buf[idx % WIN] = feat
        idx += 1

        fps_times.append(time.perf_counter())
        if len(fps_times) > 60:
            fps_times.pop(0)
        if len(fps_times) >= 2:
            measured_fps = (len(fps_times) - 1) / (fps_times[-1] - fps_times[0])

        if idx >= WIN and idx % INFER_EVERY == 0:
            slot    = idx % WIN
            ordered = np.concatenate([buf[slot:], buf[:slot]], axis=0)
            normed  = normalize_matrix(ordered)
            with torch.no_grad():
                x_t  = torch.from_numpy(normed[..., :3]).unsqueeze(0).to(device)
                m_t  = torch.from_numpy(ordered[..., 3]).unsqueeze(0).to(device)
                pred = model(x_t, m_t).cpu().numpy()[0]

            nyq = max(measured_fps / 2.0, 1e-3)
            high = min(4.0, 0.95 * nyq)
            try:
                sig = bandpass(pred, fs=measured_fps, low=0.7, high=high) if high > 0.71 else pred
            except Exception:
                sig = pred
            sig_disp = sig

            freqs = np.fft.rfftfreq(len(sig), d=1.0 / measured_fps)
            spec  = np.abs(np.fft.rfft(sig - sig.mean()))
            band  = (freqs >= 0.7) & (freqs <= 4.0)
            if band.any():
                bpm = float(freqs[band][spec[band].argmax()] * 60.0)

        h, w = frame.shape[:2]
        if idx >= WIN:
            plot_h = 120
            rng = sig_disp.max() - sig_disp.min()
            s = (sig_disp - sig_disp.min()) / (rng + 1e-8) if rng > 0 else np.zeros_like(sig_disp)
            xs = np.linspace(0, w - 1, WIN).astype(int)
            ys = (h - 1 - s * (plot_h - 1)).astype(int)
            cv2.rectangle(frame, (0, h - plot_h), (w, h), (0, 0, 0), -1)
            cv2.polylines(frame, [np.stack([xs, ys], axis=1)], False, (0, 255, 0), 1)
            cv2.putText(frame, f"{bpm:5.1f} BPM  fps={measured_fps:4.1f}",
                        (10, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
        else:
            cv2.putText(frame, f"buffering {idx}/{WIN}  fps={measured_fps:4.1f}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        cv2.imshow("rPPG preview (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Stopped by user.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)  # let macOS actually close the window
